# 02 — Data engineering : chargement, exploration, nettoyage, modèle relationnel

**Projet NutriHFr** · Étape 3 de la consigne (3a à 3e)

Le code métier est dans `src/nutrihfr/` (classes réutilisables) ; ce notebook l'utilise, montre les données et justifie les choix.

| Classe | Fichier | Rôle |
|---|---|---|
| `Inca3Loader` | `loader.py` | lecture robuste des CSV, renommage vers des noms canoniques, filtre population |
| `DataCleaner` | `quality.py` | règles de nettoyage, table des exclusions |
| `IntakeCalculator` | `intake.py` | calcul des apports |
| `NutriDatabase` | `database.py` | création et alimentation de la base SQLite |

In [ ]:
# --- Configuration commune à tous les notebooks ------------------------------
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C
from nutrihfr.viz import set_style, savefig, PALETTE, SEX_COLORS
set_style()
if C.MODE == "synthetic" and not C.raw_dir("inca3").exists():   # 1re utilisation : données de démo
    import runpy; runpy.run_path(str(ROOT / "scripts" / "make_synthetic_data.py"), run_name="__main__")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
print("Mode des données :", C.MODE, "->", C.raw_dir("inca3"))

## 1. Chargement

In [ ]:
from nutrihfr.loader import Inca3Loader, read_csv_robust

loader = Inca3Loader()
tables = loader.load()
for name, df in tables.items():
    print(f"{name:<18} {df.shape[0]:>8,} lignes  {df.shape[1]:>3} colonnes")
pd.DataFrame(loader.log)

### Les valeurs nutritionnelles de `conso-compo-alim` sont-elles « pour 100 g » ou « pour la portion consommée » ?

Test simple : si la valeur est *pour 100 g*, elle est **constante** pour un même aliment quelle que soit la quantité. Si elle est *pour la portion*, elle est **proportionnelle** à la quantité. Le résultat fixe `COMPO_EMBARQUEE_PAR` dans `config.py`.

In [ ]:
raw_conso = read_csv_robust(C.raw_dir("inca3") / C.INCA3_FILES["conso"])
inv_map = {v: k for k, v in C.INCA3_COLS["conso"].items()}       # nom canonique -> nom brut
code_col, qte_col = inv_map["code_aliment"], inv_map["quantite_g"]
nrj_col = next(k for k, v in C.INCA3_COLS["conso_nutrients"].items() if v == "energie_kcal")
g = raw_conso[raw_conso[qte_col] > 0].groupby(code_col)
cv_par_aliment = (g[nrj_col].std() / g[nrj_col].mean()).median()
corr_qte = g.apply(lambda d: d[nrj_col].corr(d[qte_col]) if len(d) > 5 else np.nan).median()
print(f"CV médian de l'énergie au sein d'un aliment : {cv_par_aliment:.3f}")
print(f"Corrélation médiane énergie ~ quantité au sein d'un aliment : {corr_qte:.3f}")
print("=> valeurs POUR 100 g" if cv_par_aliment < 0.05 else "=> valeurs POUR LA PORTION",
      f"(config actuelle : {C.COMPO_EMBARQUEE_PAR})")

## 2. Exploration et visualisation

In [ ]:
ind, co, al = tables["individus"], tables["consommations"], tables["aliments"]
print(f"{len(ind)} adultes, {ind['n_jours'].sum()} journées, {len(co):,} lignes de consommation, "
      f"{co['code_aliment'].nunique()} aliments distincts")
display(ind.groupby(["sexe", "classe_age"], observed=True).size().unstack())
ind[["age", "poids_kg", "taille_cm", "poids_enquete", "n_jours"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
q = co.loc[co["quantite_g"] > 0, "quantite_g"]
axes[0].hist(np.log10(q), bins=60, color=PALETTE[0])
axes[0].set(title="Quantités par prise (échelle log)", xlabel="log10(quantité en g)", ylabel="nombre de lignes")
per_day = co.groupby("id_jour").size()
axes[1].hist(per_day, bins=range(0, per_day.max() + 2), color=PALETTE[0])
axes[1].set(title="Nombre d'aliments par journée", xlabel="aliments", ylabel="journées")
top = (co.merge(al, how="left")["groupe"].value_counts().head(12)[::-1])
axes[2].barh(top.index, top.values, color=PALETTE[0], height=0.6)
axes[2].set(title="Groupes les plus fréquents", xlabel="nombre de prises")
plt.tight_layout(); savefig(fig, "02_exploration"); plt.show()

## 3. Valeurs manquantes (3b)

Stratégie : **pas d'imputation des consommations** (on ne « devine » pas ce qu'une personne a mangé). Règles :

| Variable | Si manquante | Justification |
|---|---|---|
| `quantite_g` | ligne écartée et tracée dans `exclusions` | impossible de calculer un apport |
| valeur de composition | apport du nutriment non calculé pour cette ligne (NaN), couverture mesurée | ne pas créer de faux zéros |
| `poids_kg` | individu conservé, mais **inclassable** pour la sous-déclaration (métabolisme de base non calculable) | l'imputation biaiserait le critère de Goldberg |
| `poids_enquete` | individu exclu | pas d'estimation pondérée possible |

In [ ]:
na = pd.concat({n: tables[n].isna().mean() * 100 for n in ("individus", "consommations", "composition")})
na = na[na > 0].rename("% manquants").to_frame()
na if len(na) else "Aucune valeur manquante"

## 4. Nettoyage

Règles appliquées par `DataCleaner` (justifiées par le rapport qualité du notebook 03) : doublons exacts, quantités négatives ou manquantes, quantités implausibles (> 1,5 kg pour un solide, > 3 kg pour une boisson, en une prise), aliments absents de la nomenclature (table d'exceptions). **Chaque ligne écartée est conservée avec son motif.**

> ⚠️ Doublons : avec les vraies données, deux prises identiques le même jour peuvent être légitimes (deux cafés). Si le fichier contient l'occasion de consommation (repas, heure), l'inclure dans la définition du doublon.

In [ ]:
from nutrihfr.quality import DataCleaner
cleaner = DataCleaner(tables)
clean = cleaner.run()
clean["composition"] = clean["composition"][clean["composition"]["code_aliment"].isin(clean["aliments"]["code_aliment"])]
cleaner.summary()

## 5. Modèle relationnel (3a)

```mermaid
erDiagram
    INDIVIDUS ||--o{ JOURNEES : "a"
    JOURNEES ||--o{ CONSOMMATIONS : "contient"
    ALIMENTS ||--o{ CONSOMMATIONS : "est consommé"
    ALIMENTS ||--o{ COMPOSITION : "a pour composition"
    NUTRIMENTS ||--o{ COMPOSITION : "mesuré dans"
    INDIVIDUS ||--o{ APPORTS_INDIVIDUELS : "a"
    NUTRIMENTS ||--o{ APPORTS_INDIVIDUELS : "concerne"
    INDIVIDUS { int id_indiv PK; text sexe; real age; real poids_enquete }
    JOURNEES { text id_jour PK; int id_indiv FK; int no_jour }
    CONSOMMATIONS { int id_conso PK; text id_jour FK; int code_aliment FK; real quantite_g }
    ALIMENTS { int code_aliment PK; text libelle; text groupe }
    NUTRIMENTS { text code_nutriment PK; text libelle; text unite }
    COMPOSITION { int code_aliment PK; text code_nutriment PK; real valeur_100g }
```

Choix : la composition est en **format long** (une ligne = un aliment × un nutriment) pour ajouter un nutriment sans modifier le schéma. Les contraintes `CHECK` (quantité ≥ 0, sexe codé, poids > 0) et les clés étrangères font respecter les règles **dans la base elle-même**. Schéma complet : `sql/schema.sql`.

In [ ]:
from nutrihfr.database import NutriDatabase
db = NutriDatabase()
db.load({**clean, "sources": pd.DataFrame(loader.sources).rename(columns={"table": "table_src"})})
display(db.counts())
print("Violations de clés étrangères :", len(db.check_integrity()))

Exemple : moyenne **pondérée** de la quantité de légumes consommée par jour, directement en SQL.

In [ ]:
sql = """
WITH par_indiv AS (
    SELECT i.id_indiv, i.sexe, i.poids_enquete,
           SUM(CASE WHEN a.groupe = 'Légumes' THEN c.quantite_g ELSE 0 END) * 1.0 / i.n_jours AS legumes_g_j
    FROM individus i
    JOIN consommations c ON c.id_indiv = i.id_indiv
    JOIN aliments a      ON a.code_aliment = c.code_aliment
    GROUP BY i.id_indiv
)
SELECT sexe,
       COUNT(*)                                             AS n,
       ROUND(AVG(legumes_g_j), 1)                           AS moyenne_non_ponderee,
       ROUND(SUM(legumes_g_j * poids_enquete) / SUM(poids_enquete), 1) AS moyenne_ponderee
FROM par_indiv GROUP BY sexe
"""
db.query(sql)

## 6. Représentativité et pondérations (3c)

INCA 3 est un échantillon aléatoire à plusieurs degrés ; certains profils répondent moins que d'autres. Les **pondérations** fournies par l'Anses redressent l'échantillon pour qu'il ressemble à la population française (âge, sexe, région, catégorie sociale…). Toutes les moyennes et tous les percentiles du projet sont donc **pondérés**.

Indicateur utile : la **taille d'échantillon effective de Kish** $n_{eff} = (\sum w)^2 / \sum w^2$, qui mesure la perte de précision due à la dispersion des poids.

In [ ]:
w = clean["individus"]["poids_enquete"]
n_eff = w.sum() ** 2 / (w ** 2).sum()
print(f"n = {len(w)}, n effectif (Kish) = {n_eff:.0f}  -> effet de plan de pondération = {len(w) / n_eff:.2f}")
print(f"Somme des poids = {w.sum():,.0f}")

struct = clean["individus"].groupby(["sexe", "classe_age"], observed=True).agg(n=("id_indiv", "size"), w=("poids_enquete", "sum"))
struct["% échantillon"] = 100 * struct["n"] / struct["n"].sum()
struct["% population (pondéré)"] = 100 * struct["w"] / struct["w"].sum()
struct[["% échantillon", "% population (pondéré)"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(w / w.mean(), bins=50, color=PALETTE[0])
ax.set(title="Dispersion des pondérations (poids / poids moyen)", xlabel="poids relatif", ylabel="individus")
savefig(fig, "02_ponderations"); plt.show()

## 7. Confidentialité (3d)

- Les données INCA sont **pseudonymisées** : un identifiant (`NOIND`) remplace l'identité ; aucune donnée directement identifiante n'est diffusée.
- Elles restent des **données de santé à caractère personnel** au sens du RGPD (une ré-identification par croisement reste théoriquement possible). Bonnes pratiques appliquées :
  - les fichiers bruts **ne sont pas versionnés** dans Git (`data/` est dans `.gitignore`), on documente l'URL de téléchargement à la place ;
  - on ne publie que des **résultats agrégés** ; le dashboard n'affiche jamais un individu, et masque les groupes de moins de 30 personnes ;
  - aucune tentative de croisement avec d'autres sources individuelles ;
  - respect des conditions d'utilisation et des références à citer indiquées dans la notice Anses.

## 8. Biais (3e)

| Biais | Description | Traitement dans le projet |
|---|---|---|
| Sélection / non-réponse | les participants diffèrent des non-participants | pondérations d'enquête (section 6) |
| Déclaration (sous-déclaration) | on oublie ou minimise certaines prises (grignotage, alcool, sucreries), plus souvent chez les personnes en surpoids | critère de Goldberg et analyses de sensibilité (notebook 05) |
| Désirabilité sociale | sur-déclaration des « bons » aliments (fruits, légumes) | cité en limite, non corrigeable ici |
| Peu de jours observés | 3 jours ne reflètent pas l'apport habituel ; la variabilité interindividuelle est surestimée | décomposition intra/inter (Q6) |
| Mémorisation | rappel de 24 h : oubli d'aliments | méthode standardisée de l'Anses ; limite |
| Composition | table figée à une version, aliments génériques | version documentée ; contrôle Atwater |

➡️ Suite : `03_qualite_et_validation.ipynb`